# Milestone 1 Cleaning Script
- We first cleaned the `unanchored_events.csv` to have a cleaned timestamp column and then merged it with the still uncleaned `noised.xes` and cleaned the rest

In [1]:
import pandas as pd
import pm4py
import numpy as np
import os
from openai import OpenAI

In [2]:
df_noised = pm4py.read_xes(
    "../data/noised.xes.gz",
    return_legacy_log_object=False
)

print(df_noised.shape)

/workspaces/ProM-Assignment-Group-C/.venv/lib/python3.12/site-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

(954012, 21)


## Clean unanchored Events / timestamp column
Coded by: Felix

In [43]:
df_unanchored = pd.read_csv("../data/unanchored_events.csv.gz", sep=";")

df_unanchored.head()

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,01.01.2016 09:51:15,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,01.01.2016 09:51:15,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2016-01-01 09:52:36.403000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
# identify different time formats
s = df_unanchored["time:timestamp"].astype(str).str.strip()

patterns = {
    "ISO yyyy-mm-dd": r"^\d{4}-\d{2}-\d{2}",
    "European dd.mm.yyyy": r"^\d{2}\.\d{2}\.\d{4}",
    "European d.mm.yyyy": r"^\d{1,2}\.\d{2}\.\d{4}",
    "US-like mm/dd/yyyy": r"^\d{1,2}/\d{1,2}/\d{4}",
    "invalid / custom strings": r"^(not_a_timestamp|9999|NaT|None|nan)$",
    "UTC" : r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z$"
}

for name, pattern in patterns.items():
    count = s.str.contains(pattern, regex=True, na=False).sum()
    print(name, count)

ISO yyyy-mm-dd 962480
European dd.mm.yyyy 239373
European d.mm.yyyy 239373
US-like mm/dd/yyyy 0
invalid / custom strings 0
UTC 193657


/tmp/ipykernel_49662/2464342000.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  count = s.str.contains(pattern, regex=True, na=False).sum()


In [45]:
def clean_unanchored_events_mixed_formats(
    df,
    timestamp_column="time:timestamp"
):
    df = df.copy()

    timestamps = df[timestamp_column].astype("string")

    parsed = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns, UTC]")

    # Format 1: German format, e.g. 31.12.2017 14:30:00
    mask_german = timestamps.str.match(
        r"^\d{2}\.\d{2}\.\d{4} \d{2}:\d{2}:\d{2}$",
        na=False
    )

    parsed.loc[mask_german] = pd.to_datetime(
        timestamps.loc[mask_german],
        format="%d.%m.%Y %H:%M:%S",
        errors="coerce",
        utc=True
    )

    # Format 2: ISO with T and Z, e.g. 2017-12-31T14:30:00.123456Z
    mask_iso_z = timestamps.str.match(
        r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z$",
        na=False
    )

    parsed.loc[mask_iso_z] = pd.to_datetime(
        timestamps.loc[mask_iso_z],
        format="%Y-%m-%dT%H:%M:%S.%fZ",
        errors="coerce",
        utc=True
    )

    # Fallback: try normal pandas parsing for unchanged/default timestamps
    mask_remaining = parsed.isna() & timestamps.notna()

    parsed.loc[mask_remaining] = pd.to_datetime(
        timestamps.loc[mask_remaining],
        errors="coerce",
        utc=True
    )

    df[timestamp_column] = parsed

    return df

In [46]:
df_cleaned_unanchored = clean_unanchored_events_mixed_formats(
    df_unanchored,
    timestamp_column="time:timestamp"
)

In [47]:
s = df_cleaned_unanchored["time:timestamp"].astype(str).str.strip()

patterns = {
    "ISO yyyy-mm-dd": r"^\d{4}-\d{2}-\d{2}",
    "European dd.mm.yyyy": r"^\d{2}\.\d{2}\.\d{4}",
    "European d.mm.yyyy": r"^\d{1,2}\.\d{2}\.\d{4}",
    "US-like mm/dd/yyyy": r"^\d{1,2}/\d{1,2}/\d{4}",
    "invalid / custom strings": r"^(not_a_timestamp|9999|NaT|None|nan)$",
    "UTC" : r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+Z$"
}

for name, pattern in patterns.items():
    count = s.str.contains(pattern, regex=True, na=False).sum()
    print(name, count)

ISO yyyy-mm-dd 1201090
European dd.mm.yyyy 0
European d.mm.yyyy 0
US-like mm/dd/yyyy 0
invalid / custom strings 0
UTC 0


/tmp/ipykernel_49662/1314539912.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  count = s.str.contains(pattern, regex=True, na=False).sum()


In [48]:
# check for null values in timestamp column
df_check = df_cleaned_unanchored.copy()

print(
    df_check.groupby("EventOrigin")["time:timestamp"]
    .apply(lambda x: x.isna().sum())
)

EventOrigin
Application    222
Offer          192
Workflow       763
Name: time:timestamp, dtype: int64


In [49]:
application_na_rows = df_check[
    (df_check["EventOrigin"] == "Application") &
    (df_check["time:timestamp"].isna())
]
application_na_rows.head()

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
16251,statechange,User_112,A_Validating,Application,ApplState_1428953723,complete,NaT,Car,New credit,Application_1462703151,5000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31740,statechange,User_27,A_Complete,Application,ApplState_1056054725,complete,NaT,Home improvement,Limit raise,Application_953097922,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
48451,statechange,User_17,A_Complete,Application,ApplState_1591021281,complete,NaT,Car,New credit,Application_1979616665,7500.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
67302,statechange,User_27,A_Accepted,Application,ApplState_1153503683,complete,NaT,Home improvement,Limit raise,Application_2122367472,30000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
70429,statechange,User_18,A_Complete,Application,ApplState_768542349,complete,NaT,Existing loan takeover,New credit,Application_1901466720,50000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:
# check whole trace of one event with missing timestamp
case_id = application_na_rows.iloc[1]["case:concept:name"]

df_check.loc[
    df_check["case:concept:name"] == case_id,
    ["case:concept:name", "concept:name", "EventOrigin", "time:timestamp", "EventID"]
]

,case:concept:name,concept:name,EventOrigin,time:timestamp,EventID
31727,Application_953097922,A_Create Application,Application,2016-01-12 11:49:40+00:00,Application_953097922
31728,Application_953097922,A_Concept,Application,2016-01-12 11:49:40+00:00,ApplState_339072701
31729,Application_953097922,W_Complete application,Workflow,2016-01-12 11:49:40.257000+00:00,Workitem_190087837
31730,Application_953097922,W_Complete application,Workflow,2016-01-12 11:49:40.260000+00:00,Workitem_640792230
31731,Application_953097922,A_Accepted,Application,2016-01-12 11:50:34+00:00,ApplState_12271477
31732,Application_953097922,W_Complete application,Workflow,2016-01-12 11:50:56.698000+00:00,Workitem_1222575764
31733,Application_953097922,W_Complete application,Workflow,2016-01-12 13:32:08.186000+00:00,Workitem_545066525
31734,Application_953097922,O_Create Offer,Offer,2016-01-12 13:36:55.669000+00:00,Offer_1441589464
31735,Application_953097922,O_Created,Offer,2016-01-12 13:36:56.900000+00:00,OfferState_1404215056
31736,Application_953097922,O_Sent (mail and online),Offer,2016-01-12 13:37:10.953000+00:00,OfferState_1593764794


In [51]:
df_check["event_order"] = range(len(df_check))

In [52]:
def impute_missing_timestamps_within_cases(
    df,
    case_column="case:concept:name",
    timestamp_column="time:timestamp",
    order_column=None
):
    df = df.copy()

    df[timestamp_column] = pd.to_datetime(
        df[timestamp_column],
        errors="coerce",
        utc=True
    )

    if order_column is None:
        df["_event_order"] = range(len(df))
        order_column = "_event_order"

    df = df.sort_values([case_column, order_column]).copy()

    previous_time = df.groupby(case_column)[timestamp_column].ffill()
    next_time = df.groupby(case_column)[timestamp_column].bfill()

    missing_mask = df[timestamp_column].isna()
    imputable_mask = missing_mask & previous_time.notna() & next_time.notna()

    df.loc[imputable_mask, timestamp_column] = (
        previous_time[imputable_mask]
        + (next_time[imputable_mask] - previous_time[imputable_mask]) / 2
    )

    df.loc[imputable_mask, "timestamp_cleaning"] = "imputed_between_neighbors"

    if "_event_order" in df.columns:
        df = df.drop(columns=["_event_order"])

    return df

In [53]:
df_cleaned_unanchored = impute_missing_timestamps_within_cases(
    df_check,
    case_column="case:concept:name",
    timestamp_column="time:timestamp",
    order_column="event_order"
)

In [54]:
df_cleaned_unanchored.loc[
    df_cleaned_unanchored["case:concept:name"] == case_id,
    ["case:concept:name", "concept:name", "EventOrigin", "time:timestamp", "timestamp_cleaning"]
]

,case:concept:name,concept:name,EventOrigin,time:timestamp,timestamp_cleaning
31727,Application_953097922,A_Create Application,Application,2016-01-12 11:49:40+00:00,NaN
31728,Application_953097922,A_Concept,Application,2016-01-12 11:49:40+00:00,NaN
31729,Application_953097922,W_Complete application,Workflow,2016-01-12 11:49:40.257000+00:00,NaN
31730,Application_953097922,W_Complete application,Workflow,2016-01-12 11:49:40.260000+00:00,NaN
31731,Application_953097922,A_Accepted,Application,2016-01-12 11:50:34+00:00,NaN
31732,Application_953097922,W_Complete application,Workflow,2016-01-12 11:50:56.698000+00:00,NaN
31733,Application_953097922,W_Complete application,Workflow,2016-01-12 13:32:08.186000+00:00,NaN
31734,Application_953097922,O_Create Offer,Offer,2016-01-12 13:36:55.669000+00:00,NaN
31735,Application_953097922,O_Created,Offer,2016-01-12 13:36:56.900000+00:00,NaN
31736,Application_953097922,O_Sent (mail and online),Offer,2016-01-12 13:37:10.953000+00:00,NaN


In [55]:
# check for null values in timestamp column
df_check = df_cleaned_unanchored.copy()

print(
    df_check.groupby("EventOrigin")["time:timestamp"]
    .apply(lambda x: x.isna().sum())
)

EventOrigin
Application    28
Offer           3
Workflow       31
Name: time:timestamp, dtype: int64


In [56]:
def fill_missing_timestamps_with_neighbor(
    df,
    case_column="case:concept:name",
    timestamp_column="time:timestamp",
    order_column=None,
    cleaning_column="timestamp_cleaning"
):
    df = df.copy()

    df[timestamp_column] = pd.to_datetime(
        df[timestamp_column],
        errors="coerce",
        utc=True
    )

    if cleaning_column not in df.columns:
        df[cleaning_column] = None

    if order_column is None:
        df["_event_order"] = range(len(df))
        order_column = "_event_order"

    df = df.sort_values([case_column, order_column]).copy()

    previous_time = df.groupby(case_column)[timestamp_column].ffill()
    next_time = df.groupby(case_column)[timestamp_column].bfill()

    missing_mask = df[timestamp_column].isna()

    previous_mask = missing_mask & previous_time.notna()
    df.loc[previous_mask, timestamp_column] = previous_time[previous_mask]
    df.loc[previous_mask, cleaning_column] = "filled_from_previous_timestamp"

    missing_mask = df[timestamp_column].isna()

    next_mask = missing_mask & next_time.notna()
    df.loc[next_mask, timestamp_column] = next_time[next_mask]
    df.loc[next_mask, cleaning_column] = "filled_from_next_timestamp"

    missing_mask = df[timestamp_column].isna()
    df.loc[missing_mask, cleaning_column] = "could_not_fill_timestamp"

    if "_event_order" in df.columns:
        df = df.drop(columns=["_event_order"])

    return df

In [57]:
df_cleaned_unanchored = fill_missing_timestamps_with_neighbor(
    df_cleaned_unanchored,
    case_column="case:concept:name",
    timestamp_column="time:timestamp"
)

In [58]:
df_cleaned_unanchored["timestamp_cleaning"].value_counts()

timestamp_cleaning
imputed_between_neighbors         1115
filled_from_previous_timestamp      34
filled_from_next_timestamp          28
Name: count, dtype: int64

In [59]:
#export cleaned unanchored events to csv
df_cleaned_unanchored.to_csv("../data/cleaned_unanchored_events.csv.gz", index=False, sep=";", compression="gzip")

## Cleaning Other Patterns

In [100]:
df_cleaned = df_noised.merge(
    df_cleaned_unanchored[["EventID", "time:timestamp"]],
    on="EventID",
    how="left",
    suffixes=("_noised", "_clean")
)

df_cleaned["time:timestamp"] = df_cleaned["time:timestamp_clean"]

df_cleaned = df_cleaned.drop(columns=["time:timestamp_noised", "time:timestamp_clean"])

df_cleaned.head()

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,...,CreditScore,OfferedAmount,OfferID,start_timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,pollution_type,time:timestamp
0,Created,User_1,A_Create Application,Application,Application_1000086665,complete,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaT,"Other, see explanation",New credit,Application_1000086665,5000.0,NaN,2016-08-03 15:57:21+00:00
1,statechange,User_1,A_Submitted,Application,ApplState_161925113,complete,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaT,"Other, see explanation",New credit,Application_1000086665,5000.0,NaN,2016-08-03 15:57:21+00:00
2,Created,User_1,W_Handle leads,Workflow,Workitem_747707399,schedule,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaT,"Other, see explanation",New credit,Application_1000086665,5000.0,NaN,2016-08-03 15:57:21.963000+00:00
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1030261128,withdraw,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaT,"Other, see explanation",New credit,Application_1000086665,5000.0,NaN,2016-08-03 15:58:28.286000+00:00
4,Created,User_1,W_Complete application,Workflow,Workitem_1127124826,schedule,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaT,"Other, see explanation",New credit,Application_1000086665,5000.0,NaN,2016-08-03 15:58:28.293000+00:00


### Clean Polluted Labels
Coded by: Felix

In [101]:
df_cleaned["concept:name"].value_counts()

concept:name
W_Call after offers                                              181252
W_Call incomplete files                                          160058
W_Complete application                                            98837
W_Handle leads                                                    44935
O_Created                                                         40830
                                                                  ...  
W_Complete application - Incident No. Application_987445453           1
W_Call after offers - Incident No. Application_988527938              1
W_Assess potential fraud - Incident No. Application_988854237         1
W_Call after offers - Incident No. Application_992509096              1
A_Accepted - Incident No. Application_998600110                       1
Name: count, Length: 1002, dtype: int64

In [102]:
def clean_polluted_labels(
    df,
    activity_column="concept:name",
    cleaning_column="label_cleaning"
):
    df = df.copy()

    if cleaning_column not in df.columns:
        df[cleaning_column] = None

    original_labels = df[activity_column].copy()

    # convert as string
    df[activity_column] = df[activity_column].astype("string")

    patterns = [
        r"\._\d+$",                              # ._1691306052
        r"_\d+$",                                # _1691306052
        r"\s*-\s*Incident No\.?\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*Incident\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*Case\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*Application\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*User\s*[A-Za-z0-9_\-]+",
        r"\s*-\s*Resource\s*[A-Za-z0-9_\-]+",
        r"\s*#\s*[A-Za-z0-9_\-]+",
        r"\s*\(\s*id\s*[:=]?\s*[A-Za-z0-9_\-]+\s*\)",
        r"\s*\(\s*case\s*[:=]?\s*[A-Za-z0-9_\-]+\s*\)",
    ]

    for pattern in patterns:
        df[activity_column] = df[activity_column].str.replace(
            pattern,
            "",
            regex=True
        )

    # normalize whitespace
    df[activity_column] = (
        df[activity_column]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    changed_mask = original_labels.astype("string") != df[activity_column]

    df.loc[changed_mask, cleaning_column] = "cleaned_polluted_label"

    return df

In [103]:
df_cleaned = clean_polluted_labels(
    df_cleaned,
    activity_column="concept:name"
)

In [104]:
df_cleaned["concept:name"].value_counts()

concept:name
W_Call after offers                           181445
W_Call incomplete files                       160222
W_Complete application                         98976
W_Handle leads                                 44977
O_Created                                      40868
O_Create Offer                                 40849
O_Sent (mail and online)                       37802
A_Accepted                                     29927
A_Concept                                      29910
A_Create Application                           29900
A_Complete                                     29795
O_Returned                                     22133
A_Incomplete                                   21908
W_Finalize application                         21291
W_Finish application                           21067
O_Cancelled                                    19818
A_Submitted                                    19404
O_Accepted                                     16362
A_Pending                        

### Clean Distorted Labels
Coded by: Felix

In [105]:
DISTORTED_LABEL_MAPPING = {
    "W": {
        "Validate appl.": "Validate application",
        "Call after ofrs.": "Call after offers",
        "Call incompl. docs.": "Call incomplete files",
        "Call incompl. files": "Call incomplete files",
        "Complete appl.": "Complete application",
        "Handle lds.": "Handle leads",
        "Assess pot. frd.": "Assess potential fraud",
        "Assess pot. fraud": "Assess potential fraud",
        "Short. compl.": "Shortened completion",
        "Short. completion": "Shortened completion",
        "Pers. Ln. coll.": "Personal Loan collection",
        "Pers. Loan collection": "Personal Loan collection",
    },
    "O": {
        "Create Off.": "Create Offer",
        "Crtd.": "Created",
        "Sent (email and onl.)": "Sent (mail and online)",
        "Sent (mail and onl.)": "Sent (mail and online)",
        "Sent (onl. only)": "Sent (online only)",
        "Ret.": "Returned",
        "Canc.": "Cancelled",
        "Acc.": "Accepted",
        "Ref.": "Refused",
    },
    "A": {
        "Valid.": "Validating",
        "Create App.": "Create Application",
        "Conc.": "Concept",
        "Concept.": "Concept",
        "Acc.": "Accepted",
        "Comp.": "Complete",
        "Incompl.": "Incomplete",
        "Subm.": "Submitted",
        "Pend.": "Pending",
        "Canc.": "Cancelled",
        "Den.": "Denied",
    }
}

In [106]:
def build_label_mapping(grouped_mapping):
    mapping = {}

    for prefix, labels in grouped_mapping.items():
        for distorted, cleaned in labels.items():
            mapping[f"{prefix}_{distorted}"] = f"{prefix}_{cleaned}"

    return mapping

In [107]:
label_mapping = build_label_mapping(DISTORTED_LABEL_MAPPING)

In [108]:
label_mapping

{'W_Validate appl.': 'W_Validate application',
 'W_Call after ofrs.': 'W_Call after offers',
 'W_Call incompl. docs.': 'W_Call incomplete files',
 'W_Call incompl. files': 'W_Call incomplete files',
 'W_Complete appl.': 'W_Complete application',
 'W_Handle lds.': 'W_Handle leads',
 'W_Assess pot. frd.': 'W_Assess potential fraud',
 'W_Assess pot. fraud': 'W_Assess potential fraud',
 'W_Short. compl.': 'W_Shortened completion',
 'W_Short. completion': 'W_Shortened completion',
 'W_Pers. Ln. coll.': 'W_Personal Loan collection',
 'W_Pers. Loan collection': 'W_Personal Loan collection',
 'O_Create Off.': 'O_Create Offer',
 'O_Crtd.': 'O_Created',
 'O_Sent (email and onl.)': 'O_Sent (mail and online)',
 'O_Sent (mail and onl.)': 'O_Sent (mail and online)',
 'O_Sent (onl. only)': 'O_Sent (online only)',
 'O_Ret.': 'O_Returned',
 'O_Canc.': 'O_Cancelled',
 'O_Acc.': 'O_Accepted',
 'O_Ref.': 'O_Refused',
 'A_Valid.': 'A_Validating',
 'A_Create App.': 'A_Create Application',
 'A_Conc.': 'A_Con

In [109]:
def clean_distorted_labels(
    df,
    activity_column="concept:name"
):
    df = df.copy()

    label_mapping = build_label_mapping(DISTORTED_LABEL_MAPPING)

    df[activity_column] = df[activity_column].replace(label_mapping)

    return df

In [110]:
df_cleaned = clean_distorted_labels(df_cleaned)

In [111]:
df_cleaned["concept:name"].value_counts()

concept:name
W_Call after offers                           191092
W_Call incomplete files                       168529
W_Complete application                        106542
W_Handle leads                                 47264
O_Create Offer                                 42995
O_Created                                      42995
O_Sent (mail and online)                       39707
A_Create Application                           31509
A_Concept                                      31509
A_Accepted                                     31509
A_Complete                                     31362
O_Returned                                     23305
A_Incomplete                                   23055
W_Finalize application                         21291
W_Finish application                           21067
O_Cancelled                                    20898
A_Submitted                                    20423
O_Accepted                                     17228
A_Pending                        

### Clean Scattered Cases
Coded by: Jonas

In [112]:
# read CSV containing the scattered rows
validation_df = pd.read_csv("../data/scattered_events.csv.gz", sep=",")

In [113]:
# reindex
validation_df = validation_df.reindex(columns=df_cleaned.columns)

# merging the two dataframes
df_cleaned = pd.concat([df_cleaned, validation_df], ignore_index=True)

# parse timestamps
df_cleaned["time:timestamp"] = pd.to_datetime(
    df_cleaned["time:timestamp"],
    #errors="coerce",
    utc=True
)

### Clean Synonymous Labels
Coded by: Thien

In [116]:
ENDPOINT = os.getenv("OPENAI_ENDPOINT")
DEPLOYMENT_NAME = os.getenv("OPENAI_DEPLOYMENT_NAME")
API_KEY = os.getenv("OPENAI_API_KEY")

openai_client = OpenAI(
    base_url=ENDPOINT,
    api_key=API_KEY,
)

In [117]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [119]:
def find_similar_labels(
    df: pd.DataFrame,
    threshold: float = 0.65,
    prefix_filter: bool = True
) -> pd.DataFrame:
    """
    Identifies semantically similar event labels using OpenAI embeddings.

    This function computes embeddings for all unique values in the
    'concept:name' column of the input DataFrame and calculates pairwise
    cosine similarity between them. Pairs with similarity above the given
    threshold are returned as potential synonym candidates.

    Optionally, labels can be filtered by prefix (e.g., 'W_', 'A_') to
    avoid comparing activities from different process sections.

    Args:
        df (pd.DataFrame): Input event log dataframe with a 'concept:name' column.
        threshold (float): Similarity threshold for considering two labels as synonyms.
        prefix_filter (bool): If True, only compare labels with the same prefix.

    Returns:
        pd.DataFrame: DataFrame containing pairs of similar labels and their similarity scores.
    """
    labels = df["concept:name"].unique()

    # --- embeddings ---
    embeddings = []
    for l in labels:
        res = openai_client.embeddings.create(
            model=DEPLOYMENT_NAME,
            input=l
        )
        embeddings.append(res.data[0].embedding)
    embeddings = np.array(embeddings)

    # --- similarity pairs ---
    results = []
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            if prefix_filter:
                if labels[i].split("_")[0] != labels[j].split("_")[0]:
                    continue
            sim = cosine_similarity(embeddings[i], embeddings[j])
            if sim > threshold:
                results.append({
                    "label_1": labels[i],
                    "label_2": labels[j],
                    "similarity": sim
                })
    return pd.DataFrame(results)

In [120]:
result = find_similar_labels(df_cleaned, threshold=0.65, prefix_filter = True)
result

,label_1,label_2,similarity
0,W_Complete application,W_Finalize application,0.667781
1,O_Sent (mail and online),O_Sent (online only),0.902907
2,W_Finish application,W_Finalize application,0.746618
3,W_Precheck application: Applicant Identity,W_Precheck application: Form Completeness,0.708304
4,W_Precheck application: Applicant Identity,W_Precheck application: Credit History,0.712661
5,W_Precheck application: Form Completeness,W_Precheck application: Credit History,0.663201


In [121]:
#Use domain knowledge to exclude label pairs with high similarity they don't make sense
SYNONYMOUS_LABELS = ["W_Complete application", "W_Finish application", "W_Finalize application"]
df_cleaned[df_cleaned["concept:name"].isin(SYNONYMOUS_LABELS)]["concept:name"].value_counts()

concept:name
W_Complete application    106542
W_Finalize application     21291
W_Finish application       21067
Name: count, dtype: int64[pyarrow]

In [122]:
def replace_with_most_frequent(
    df: pd.DataFrame,
    synonym_list: list,
    column: str = "concept:name"
) -> pd.DataFrame:
    """
    Replaces a group of synonymous event labels with the most frequent label.

    This function identifies the most common label within a given list of
    synonymous activities and replaces all occurrences of those labels in the
    DataFrame with the most frequent one.

    Args:
        df (pd.DataFrame): Input event log dataframe.
        synonym_list (list): List of synonymous labels to be replaced.
        column (str): Name of the column containing the labels (default is 'concept:name').
        
    Returns:
        pd.DataFrame: Copy of the dataframe with replaced labels.
    """
    df = df.copy()
    subset = df[df[column].isin(synonym_list)]
    most_frequent = subset[column].value_counts().idxmax()
    df.loc[df[column].isin(synonym_list), column] = most_frequent
    return df

In [126]:
df_cleaned = replace_with_most_frequent(df_cleaned, SYNONYMOUS_LABELS)
df_cleaned[df_cleaned["concept:name"].isin(SYNONYMOUS_LABELS)]["concept:name"].value_counts()

concept:name
W_Complete application    148900
Name: count, dtype: int64[pyarrow]

### Clean Colletaral Events
Coded by: Thien

In [127]:
def detect_collateral_events(
    df: pd.DataFrame,
    time_window: str = "5s",
    similarity: bool = False,
    similarity_df: pd.DataFrame = None,
    threshold: float = None
):
    """
    Detects potential collateral events based on temporal proximity
    and optional semantic similarity filtering.

    Args:
        df (pd.DataFrame): Input event log dataframe with 'case:concept:name', 'concept:name', and 'time:timestamp' columns.
        time_window (str): Maximum time difference between events to be considered collateral (e.g., '5s', '1m').
        similarity (bool): If True, apply semantic similarity filtering based on provided similarity_df.
        similarity_df (pd.DataFrame): DataFrame containing pairs of labels and their similarity scores (required if similarity=True).
        threshold (float): Minimum similarity score to consider labels related (required if similarity=True).
    
    Returns:
        pd.DataFrame: DataFrame containing pairs of potentially collateral events with their time difference and similarity score (if applicable).
    """

    df = df.copy()
    df = df.sort_values(["case:concept:name", "time:timestamp"])

    # --- build allowed similarity set ---
    allowed_pairs = set()
    if similarity and similarity_df is not None:
        sim_filtered = similarity_df[similarity_df["similarity"] >= threshold]
        for _, row in sim_filtered.iterrows():
            allowed_pairs.add((row["label_1"], row["label_2"]))
            allowed_pairs.add((row["label_2"], row["label_1"]))  # symmetric

    results = []
    for case_id, group in df.groupby("case:concept:name"):
        group = group.sort_values("time:timestamp")
        for i in range(len(group) - 1):
            row_a = group.iloc[i]
            row_b = group.iloc[i + 1]
            time_diff = row_b["time:timestamp"] - row_a["time:timestamp"]
            if time_diff <= pd.Timedelta(time_window):
                e1 = row_a["concept:name"]
                e2 = row_b["concept:name"]

                # --- semantic filter (optional) ---
                if similarity:
                    if (e1, e2) not in allowed_pairs:
                        continue
                results.append({
                    "case": case_id,
                    "event_1": e1,
                    "event_2": e2,
                    "time_diff": time_diff
                })

    return pd.DataFrame(results)

In [128]:
result_for_collateral = result[~result['label_1'].isin(SYNONYMOUS_LABELS)]
collateral_df = detect_collateral_events(df_cleaned, time_window="5s", similarity=True, similarity_df=result_for_collateral, threshold=0.65)

In [129]:
set(collateral_df['event_1'].unique()) | set(collateral_df['event_2'].unique())

{'W_Precheck application: Applicant Identity',
 'W_Precheck application: Credit History',
 'W_Precheck application: Form Completeness'}

In [130]:
def merge_collateral_events(df, merge_dict, time_col="time:timestamp"):
    """
    Merges collateral sub-events into a single canonical event.

    For each group of events in merge_dict values:
    - All matching rows are replaced by the key label
    - Other attributes are taken from the earliest event in time

    Args:
        df (pd.DataFrame): Input event log dataframe.
        merge_dict (dict): Dictionary where keys are canonical labels and values are lists of sub-event labels to be merged.
        time_col (str): Name of the timestamp column to determine event order (default is 'time:timestamp').
    
    Returns:
        pd.DataFrame: DataFrame with collateral events merged into canonical events.
    """
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])
    rows_to_drop = []
    rows_to_add = []

    for canonical, subevents in merge_dict.items():
        # process per case
        for case_id, group in df.groupby("case:concept:name"):
            subset = group[group["concept:name"].isin(subevents)]
            if subset.empty:
                continue
            # earliest event per case
            anchor = subset.sort_values(time_col).iloc[0]
            new_row = anchor.copy()
            new_row["concept:name"] = canonical
            rows_to_add.append(new_row)
            rows_to_drop.extend(subset.index.tolist())

    df = df.drop(index=rows_to_drop)
    df = pd.concat([df, pd.DataFrame(rows_to_add)], ignore_index=True)
    df = df.sort_values(["case:concept:name", time_col]).reset_index(drop=True)
    print("Rows to drop:", len(rows_to_drop))
    print("Rows added:", len(rows_to_add))
    return df

In [131]:
COLLATERAL_LABELS_DICT = {
  "W_Precheck application": [
    "W_Precheck application: Applicant Identity",
    "W_Precheck application: Form Completeness",
    "W_Precheck application: Credit History"
  ]
}
df_cleaned_final = merge_collateral_events(df_cleaned, COLLATERAL_LABELS_DICT)

Rows to drop: 57
Rows added: 19


## Data Quality Statistics
Coded by: Jonas

In [139]:
def print_stats(df):
    print(f"traces: {df['case:concept:name'].nunique()}")
    print(f"events: {len(df)}")
    print(f"distinct activities: {df['concept:name'].nunique()}")

    # ensure correct ordering for trace reconstruction
    df_sorted = df.copy()

    df_sorted["time:timestamp"] = pd.to_datetime(
        df_sorted["time:timestamp"],
        utc=True,
        errors="coerce"
    )

    df_sorted = df_sorted.sort_values(["case:concept:name", "time:timestamp"])

    # distinct traces / variants
    traces = []

    for _, group in df_sorted.groupby("case:concept:name"):
        trace = tuple(group["concept:name"].astype(str).tolist())
        traces.append(trace)

    distinct_traces = len(set(traces))

    # trace lengths
    trace_lengths = df_sorted.groupby("case:concept:name").size()

    print(f"distinct traces: {distinct_traces}")
    print(f"average trace length: {trace_lengths.mean():.2f}")
    print(f"min trace length: {trace_lengths.min()}")
    print(f"max trace length: {trace_lengths.max()}")

    # log duration
    log_duration = df_sorted["time:timestamp"].max() - df_sorted["time:timestamp"].min()
    print(f"log duration: {log_duration}")

    # missing values
    missing_values = df.isna().sum().sum()
    print(f"missing values: {missing_values}")

In [142]:
log_clean = pm4py.read_xes("../data/BPI Challenge 2017.xes.gz")
df_clean = pm4py.convert_to_dataframe(log_clean)

parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

In [144]:
print("Statistics for clean log")
print_stats(df_clean)

print("\nStatistics for noised log")
print_stats(df_noised)

print("\nStatistics for recoverd log")
print_stats(df_cleaned_final)

Statistics for clean log
traces: 31509
events: 1202267
distinct activities: 26
distinct traces: 15930
average trace length: 38.16
min trace length: 10
max trace length: 180
log duration: 397 days 04:19:48.195000
missing values: 9166317

Statistics for noised log
traces: 31509
events: 954012
distinct activities: 1002
distinct traces: 27843
average trace length: 30.28
min trace length: 10
max trace length: 146
log duration: 397 days 04:17:48.027000
missing values: 8086592

Statistics for recoverd log
traces: 31509
events: 1202286
distinct activities: 27
distinct traces: 16471
average trace length: 38.16
min trace length: 10
max trace length: 180
log duration: 397 days 04:19:48.499000
missing values: 11770466


## Export Recovered XES

In [145]:
event_log_recovered = pm4py.convert_to_event_log(df_cleaned_final)

In [146]:
pm4py.write_xes(event_log_recovered, "../data/recovered.xes.gz")

exporting log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]